# RNN Proj

In [147]:
import torch
import pandas as pd
import requests
import re
from bs4 import BeautifulSoup
import tokenizers
from torch.utils.data import DataLoader, Dataset
import numpy as np

device = "cuda:1"

## Step 1: Data Aggregation

In [97]:
text_strs_train = []
text_strs_valid = []

response = requests.get("https://classics.mit.edu/Caesar/gallic.mb.txt")
text = response.text.split("BOOK 1")[1]
cleaned = re.sub(r'Chapter \d+', '', text)
cleaned = cleaned.replace("\n", " ")
text_strs_train.append(cleaned)

In [98]:
response = requests.get("https://gutenberg.org/cache/epub/57493/pg57493.txt")
soup = BeautifulSoup(response.text, "html.parser")
text = soup.get_text()
text = text.split("BOOK I.[34]")[1].split("END OF VOL. I.")[0]
cleaned = re.sub(r'\[\d+\]', '', text)
cleaned = cleaned.replace("\n", " ")
cleaned = cleaned.replace("\r", " ")
text_strs_train.append(cleaned)

In [99]:
response = requests.get("https://gutenberg.org/cache/epub/60230/pg60230.txt")
soup = BeautifulSoup(response.text, "html.parser")
text = soup.get_text()
text = text.split("PEOPLES WHO NOW EXIST, OR FORMERLY EXISTED.")[1].split("FOOTNOTES:")[0]
cleaned = re.sub(r'\[\d+\]', '', text)
cleaned = cleaned.replace("\n", " ")
cleaned = cleaned.replace("\r", " ")
text_strs_train.append(cleaned)

In [100]:
response = requests.get("https://gutenberg.org/cache/epub/59131/pg59131.txt")
soup = BeautifulSoup(response.text, "html.parser")
text = soup.get_text()
text = text.split("CHAP. 1. (1.)—THE EXTREME SMALLNESS OF INSECTS.")[1].split("FOOTNOTES:")[0]
cleaned = re.sub(r'\[\d+\]', '', text)
cleaned = cleaned.replace("\n", " ")
cleaned = cleaned.replace("\r", " ")
text_strs_train.append(cleaned)

In [101]:
response = requests.get("https://gutenberg.org/cache/epub/61113/pg61113.txt")
soup = BeautifulSoup(response.text, "html.parser")
text = soup.get_text()
text = text.split("CHAP. 1. (1.)—TASTE OF THE ANCIENTS FOR AGRICULTURE.")[1].split("FOOTNOTES:")[0]
cleaned = re.sub(r'\[\d+\]', '', text)
cleaned = cleaned.replace("\n", " ")
cleaned = cleaned.replace("\r", " ")
text_strs_valid.append(cleaned)

In [102]:
response = requests.get("https://gutenberg.org/cache/epub/60688/pg60688.txt")
soup = BeautifulSoup(response.text, "html.parser")
text = soup.get_text()
text = text.split("THE REMEDIES DERIVED FROM THE FOREST TREES.")[1].split("FOOTNOTES:")[0]
cleaned = re.sub(r'\[\d+\]', '', text)
cleaned = cleaned.replace("\n", " ")
cleaned = cleaned.replace("\r", " ")
text_strs_train.append(cleaned)

In [103]:
response = requests.get("https://gutenberg.org/cache/epub/62704/pg62704.txt")
soup = BeautifulSoup(response.text, "html.parser")
text = soup.get_text()
text = text.split("ECHENEÏS: TWO REMEDIES.")[1].split("FOOTNOTES:")[0]
cleaned = re.sub(r'\[\d+\]', '', text)
cleaned = cleaned.replace("\n", " ")
cleaned = cleaned.replace("\r", " ")
text_strs_train.append(cleaned)

## Step 2: Tokenizing

In [ ]:
bpe_model = tokenizers.models.BPE(unk_token="<unk>")
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
special_tokens = ["<unk>", "<pad>", "<sos>", "<eos>"]
bpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=2000, special_tokens=special_tokens)
train_data = [t.lower() for t in text_strs_train]
bpe_tokenizer.train_from_iterator(train_data, bpe_trainer)

In [209]:
training_sets = []
for book in text_strs_train:
    sentences = [i.strip() for i in book.split(".") if len(i.strip())>20]
    for sentence in sentences:
        spaces = [i for i, c in enumerate(sentence) if c == ' ']
        try:
            mid = spaces[int(len(spaces)*np.random.choice(np.arange(.4,.61,.01)))]
        except:
            continue
        x = sentence[:mid]
        y = sentence[mid:]
        training_sets.append((x,y))

In [210]:
class RomeDataset(Dataset):
     def __init__(self, text, window_length):
        self.encoded_text = [bpe_tokenizer.encode(t.lower()).ids for t in text]
        self.window_length = window_length
        self.windows = []
        for text in self.encoded_text:
            for start_idx in range(0, len(text)-window_length-1):
                self.windows.append(text[start_idx:start_idx+window_length+1])
    # def __getitem__(self):
    
        

### TODO:
- Fix Dataset such that it handles input, target
- DataLoaders
- The actual model
- Testing